## 1. 导入依赖库
在这个教程中，我们将使用 `cobra` 库来加载和处理基因组规模代谢模型（GEM），并使用 `dGbyG.api` 中的 `predict_transformed_dG_prime_for_GEM` 函数来预测模型中代谢物和反应的转化标准吉布斯自由能（$\Delta_r G'^{\circ}$）。


In [1]:
import cobra
from dGbyG.api import predict_transformed_dG_prime_for_GEM

## 2. 定义细胞区室的环境条件
转化标准吉布斯自由能的计算高度依赖于反应发生的微环境。在此步骤中，我们为 GEM 中的各个细胞区室定义了具体的物理化学参数，包括：
- **pH**: 区室的酸碱度
- **I**: 离子强度
- **e_potential**: 电势

这些参数将用于后续的热力学校正计算。


In [2]:
compartment_conditions = {
    "Cytosol": {"pH": 7.2, "I": 0.25, "e_potential": 0.000},
    "Extracellular": {"pH": 7.4, "I": 0.25, "e_potential": 0.030},
    "Nucleus": {"pH": 7.2, "I": 0.25, "e_potential": 0.000},
    "Endoplasmic reticulum": {"pH": 7.2, "I": 0.25, "e_potential": 0.000},
    "Golgi apparatus": {"pH": 6.3, "I": 0.25, "e_potential": 0.000},
    "Lysosome": {"pH": 5.5, "I": 0.25, "e_potential": 0.019},
    "Mitochondria": {"pH": 8.0, "I": 0.25, "e_potential": -0.155},
    "Inner mitochondria": {"pH": 7.0, "I": 0.25, "e_potential": 0.000},
    "Peroxisome": {"pH": 7.0, "I": 0.25, "e_potential": 0.012}
}

## 3. 加载基因组规模代谢模型（GEM）
我们使用 `cobra.io.read_sbml_model` 函数从本地 XML 文件中读取 GEM 模型。请确保模型文件路径 `gem_path` 指向正确的文件位置。加载完成后，模型对象 `gem` 将包含所有的代谢物、反应及其注释信息。


In [3]:
gem_path = './Mouse-GEM.xml'
gem = cobra.io.read_sbml_model(gem_path)

https://identifiers.org/taxonomy/ does not conform to 'http(s)://identifiers.org/collection/id' or'http(s)://identifiers.org/COLLECTION:id


## 4. 检查代谢物结构标识符
`dGbyG` 依赖于代谢物的外部数据库标识符来映射并查找其化学结构（如 InChI、SMILES），进而进行 pKa 和热力学参数的计算。此代码块遍历模型中所有代谢物的注释，汇总并打印出模型中包含的所有 CID 类型。这有助于我们对比模型拥有的标识符与 `Compound.recognizable_cids` 中的可识别标识符，从而确认哪些类型可用于结构映射，并为后续设置 `use_met_id_types` 和 `ignore_met_id_types` 参数提供依据。


In [4]:
x = set()
for met in gem.metabolites:
    for cid_type, cid in met.annotation.items():
        x.add(cid_type)
print('CID types: ',x)

CID types:  {'kegg.compound', 'metanetx.chemical', 'lipidmaps', 'pubchem.compound', 'chebi', 'bigg.metabolite', 'sbo', 'hmdb', 'inchi', 'vmhmetabolite'}


## 5. 批量预测转化标准吉布斯自由能
这是教程的核心步骤。我们调用 `predict_transformed_dG_prime_for_GEM` 函数，传入以下参数：
- `gem`: 加载的代谢模型对象。
- `compartment_conditions`: 前面定义的细胞区室环境条件字典，支持缩写或全名匹配。
- `use_met_id_types`: 指定使用哪些注释类型来构建化合物对象。默认为 `'all'`，即使用所有可识别的标识符类型。如果只需要使用部分标识符，可以传入一个列表（如 `['bigg.metabolite', 'chebi']`），此时 `ignore_met_id_types` 将不生效。
- `ignore_met_id_types`: 当 `use_met_id_types='all'` 时，需要忽略的注释类型列表。匹配规则为前缀匹配（不区分大小写）。这里我们忽略 `'name'` 和 `'inchi_key'`，因为这些标识符需要联网查询，速度较慢。

该函数会自动处理代谢物的 pKa 预测、标准吉布斯自由能计算以及基于环境条件的转化校正，最终返回代谢物（`Met_df`）和反应（`Rxn_df`）的热力学预测结果。


In [ ]:
# 示例：仅使用 bigg.metabolite 和 chebi 作为标识符进行预测（注意：指定 use_met_id_types 后 ignore_met_id_types 将不生效）
# Met_df, Rxn_df = predict_transformed_dG_prime_for_GEM(gem, compartment_conditions=compartment_conditions, use_met_id_types=['bigg.metabolite', 'chebi'])

# 实际用法：使用所有可识别的标识符，但忽略需要联网查询的 name 和 inchi_key 以加快速度
Met_df, Rxn_df = predict_transformed_dG_prime_for_GEM(gem, compartment_conditions=compartment_conditions, ignore_met_id_types=['name', 'inchi_key'])

Processing metabolites: 100%|██████████| 8454/8454 [00:53<00:00, 159.12it/s]


✅ All molecules exist in the pKa database — skipping prediction.


Predicting transformed standard Gibbs free energy for metabolites: 100%|██████████| 8454/8454 [24:14<00:00,  5.81it/s] 
Predicting transformed standard Gibbs free energy for reactions: 100%|██████████| 12987/12987 [01:26<00:00, 149.85it/s] 


## 6. 查看代谢物预测结果
使用 `Met_df.head()` 查看代谢物的预测结果。返回的 DataFrame 结构如下：
- **行索引**：模型中的代谢物 ID。
- **列名**：不同的数据库注释来源。
- **单元格值**：元组格式为 `(预测的转化标准吉布斯自由能, 标准差)`。

对于缺失数据的表示，需要注意区分以下两种情况：
- `(nan, nan)`：表示该代谢物拥有此类型的 ID 注释，但无法根据该 ID 获取到有效的化学结构信息。
- `NaN`：表示该代谢物在模型中根本未提供此类型的 ID 注释。


In [9]:
Met_df.head()

,bigg.metabolite,chebi,kegg.compound,metanetx.chemical,hmdb,pubchem.compound,lipidmaps,inchi
MAM00001c,"(718.0063760200995, 9.5092134475708)","(718.0063760200995, 9.50921630859375)","(718.0063760200995, 9.50921630859375)","(nan, nan)",NaN,NaN,NaN,NaN
MAM00001e,"(736.2622027237875, 9.509214401245117)","(736.2622027237875, 9.50921630859375)","(736.2622027237875, 9.50921630859375)","(nan, nan)",NaN,NaN,NaN,NaN
MAM00002c,"(nan, nan)","(850.2859780708899, 12.084856986999512)","(850.2859780708899, 12.084856986999512)","(nan, nan)","(850.2859780708899, 12.084856033325195)","(850.2859780708899, 12.08484935760498)",NaN,NaN
MAM00002e,"(nan, nan)","(868.5418047745924, 12.084853172302246)","(868.5418047745924, 12.084856986999512)","(nan, nan)","(868.5418047745924, 12.084855079650879)","(868.5418047745924, 12.084856986999512)",NaN,NaN
MAM00003c,NaN,"(1123.998093883676, 4.011264324188232)",NaN,"(nan, nan)",NaN,NaN,"(1123.998093883676, 4.011266708374023)",NaN


## 7. 查看反应预测结果
使用 `Rxn_df.head()` 查看反应的预测结果。返回的 DataFrame 中：
- 行索引为模型中的反应 ID
- `dGr_prime`: 反应的转化标准吉布斯自由能预测值
- `SD of dGr_prime`: 预测值的标准差


In [10]:
Rxn_df.head(5)

,dGr_prime,SD of dGr_prime
MAR03905,20.031905,0.855806
MAR03907,20.031880,0.855823
MAR04097,-5.813766,0.424463
MAR04099,-5.731635,0.424455
MAR04108,-21.468129,3.954923
